In [ ]:
!pip install groq pandas
!pip install sqlparse groq pandas

import sqlite3
import pandas as pd
from groq import Groq
import sqlparse
import json
import re

In [ ]:
# SOSTITUISCI con il nome del file che hai caricato
db_path = "concert_singer.sqlite"

def esegui_query(sql_query, path):
    try:
        conn = sqlite3.connect(path)
        # pd.read_sql_query trasforma il risultato del DB in una tabella facile da leggere
        risultato = pd.read_sql_query(sql_query, conn)
        conn.close()
        return risultato
    except Exception as e:
        return f"Errore: {e}"

# PROVA: Vediamo se legge le tabelle
test_query = "SELECT name FROM sqlite_master WHERE type='table';"
print("Tabelle trovate nel database:")
print(esegui_query(test_query, db_path))

In [ ]:
# Inizializziamo il collegamento
# Incolla la tua chiave gsk_... tra le virgolette
client = Groq(api_key="INSERISCI_QUI_LA_TUA_CHIAVE")

def chiedi_al_modello(istruzione, domanda):
    """
    Questa funzione invia una domanda all'IA e restituisce la risposta.
    """
    completamento = client.chat.completions.create(
        messages=[
            {"role": "system", "content": istruzione}, # Qui diciamo all'IA come comportarsi
            {"role": "user", "content": domanda}        # Qui facciamo la domanda vera e propria
        ],
        model="llama-3.3-70b-versatile", # Usiamo Llama 3
    )
    return completamento.choices[0].message.content

# PROVA: Vediamo se risponde
print(chiedi_al_modello("Sei un assistente utile.", "Ciao! Chi sei?"))


In [ ]:
def get_oracle_tables(sql_query):
    # Pulisce e analizza la query SQL Gold per trovare i nomi delle tabelle
    parsed = sqlparse.parse(sql_query)[0]
    tables = []
    # Cerchiamo i nomi dopo la parola 'FROM' o 'JOIN'
    extract = False
    for token in parsed.tokens:
        if token.ttype is sqlparse.tokens.Keyword and token.value.upper() in ['FROM', 'JOIN']:
            extract = True
        elif extract and not token.is_whitespace:
            # Prende il nome della tabella, gestisce anche alias
            name = token.value.split(' ')[0].split(' as ')[0].replace('"', '').replace('`', '')
            if name and name.upper() not in ['SELECT', 'WHERE', 'GROUP', 'ORDER', 'LIMIT']:
                tables.append(name.lower())
            extract = False
    return list(set(tables))

def get_db_schema_details(db_path, tables):
    # Per ogni tabella oracle, legge i nomi delle colonne
    schema_text = ""
    for table in tables:
        try:
            info = esegui_query(f"PRAGMA table_info({table});", db_path)
            if isinstance(info, pd.DataFrame):
                cols = ", ".join(info['name'].tolist())
                schema_text += f"Tabella {table} (colonne: {cols}); "
        except:
            continue
    return schema_text


In [ ]:
def pipeline_text_to_sql(domanda_naturale, db_path, oracle_tables):
    # Usiamo solo le colonne delle tabelle rilevanti (Oracle)
    schema_info = get_db_schema_details(db_path, oracle_tables)

    istruzione = f"""
    Sei un esperto di database SQLite.
    Usa solo queste tabelle e colonne: {schema_info}
    Rispondi SOLO con la query SQL, senza spiegazioni e senza markdown.
    """

    query_generata = chiedi_al_modello(istruzione, domanda_naturale)
    query_pulita = query_generata.replace("```sql", "").replace("```", "").strip()

    return esegui_query(query_pulita, db_path)


In [ ]:
def pipeline_direct_table_qa(domanda_naturale, nome_tabella, db_path):
    # 1. Estraiamo TUTTI i dati dalla tabella interessata per darli all'IA
    # In un caso reale useremmo solo le tabelle "oracle relevant", come dice l'assignment
    query_estrai_tutto = f"SELECT * FROM {nome_tabella};"
    dati_tabella = esegui_query(query_estrai_tutto, db_path)

    # Trasformiamo la tabella DataFrame in una stringa di testo
    testo_tabella = dati_tabella.to_string(index=False)

    # 2. Prepariamo il prompt con i dati dentro
    istruzione = f"""
    Sei un assistente che analizza dati tabellari.
    Ecco il contenuto della tabella '{nome_tabella}':

    {testo_tabella}

    Rispondi alla domanda dell'utente basandoti solo sui dati sopra.
    Fornisci solo il dato finale (es. un numero o un nome), in modo che sia facile da leggere per un computer.
    """

    # 3. Chiediamo la risposta all'IA
    risposta_diretta = chiedi_al_modello(istruzione, domanda_naturale)

    return risposta_diretta

# PROVA
domanda_test = "Quanti cantanti ci sono?"
print("Risposta dalla Pipeline Direct Table QA:")
print(pipeline_direct_table_qa(domanda_test, "singer", db_path))


In [ ]:
with open('merged_queries.json', 'r', encoding='utf-8-sig') as f:
    spider_data = json.load(f)

domande_progetto = [q for q in spider_data if q['db_id'] == 'concert_singer']

for i in range(min(2, len(domande_progetto))):
    print(f"Domanda {i+1}: {domande_progetto[i]['question']}")
    print(f"SQL Corretto: {domande_progetto[i]['query']}\n")


In [ ]:
# Selezioniamo 5 domande per ogni database
domande_concert = [q for q in spider_data if q['db_id'] == 'concert_singer'][:5]
domande_student = [q for q in spider_data if q['db_id'] == 'student_1'][:5]
test_set = domande_concert + domande_student

risultati_test = []

for i, item in enumerate(test_set):
    domanda = item['question']
    db_file = f"{item['db_id']}.sqlite"
    sql_gold = item['query']

    # Identifichiamo le tabelle Oracle
    oracle_tables = get_oracle_tables(sql_gold)

    # 1. ORO
    risultato_oro = esegui_query(sql_gold, db_file)

    # 2. Pipeline 1: con tabelle e colonne oracle
    try:
        risultato_sql = pipeline_text_to_sql(domanda, db_file, oracle_tables)
    except:
        risultato_sql = "Errore"

    # 3. Pipeline 2: passiamo la prima tabella oracle come contesto
    # se ci sono più tabelle, le uniamo (Serializzazione)
    tabella_per_qa = oracle_tables[0] if oracle_tables else ""
    try:
        risultato_qa = pipeline_direct_table_qa(domanda, tabella_per_qa, db_file)
    except:
        risultato_qa = "Errore"

    risultati_test.append({
        "ID": i+1,
        "Domanda": domanda,
        "Oro": risultato_oro,
        "Pred_SQL": risultato_sql,
        "Pred_QA": risultato_qa
    })

# Creiamo il DataFrame finale per vedere i risultati
df_risultati = pd.DataFrame(risultati_test)
print("\n--- TEST COMPLETATI ---")
df_risultati.head(10)


In [ ]:
def normalize_string(s):
    """Trasforma tutto (anche DataFrame) in testo e pulisce per il confronto"""
    if s is None: return ""
    # Se è un DataFrame, lo trasformiamo in stringa
    if isinstance(s, pd.DataFrame):
        s = s.to_string(index=False)

    if str(s) == "Errore": return ""

    s = str(s).lower()
    # Estrae solo parole e numeri
    words = re.findall(r'\b\w+\b', s)
    return set(words)

def calcola_metrics(oro, pred):
    set_oro = normalize_string(oro)
    set_pred = normalize_string(pred)

    if not set_oro: return 0.0, 0.0
    if not set_pred: return 0.0, 0.0

    intersezione = set_oro.intersection(set_pred)

    precision = len(intersezione) / len(set_pred) if len(set_pred) > 0 else 0
    recall = len(intersezione) / len(set_oro) if len(set_oro) > 0 else 0

    return precision, recall

# Calcoliamo le medie con gestione sicura dei dati
metrics_sql = [calcola_metrics(r['Oro'], r['Pred_SQL']) for r in risultati_test]
metrics_qa = [calcola_metrics(r['Oro'], r['Pred_QA']) for r in risultati_test]

avg_prec_sql = sum(p for p, r in metrics_sql) / len(risultati_test)
avg_rec_sql = sum(r for p, r in metrics_sql) / len(risultati_test)

avg_prec_qa = sum(p for p, r in metrics_qa) / len(risultati_test)
avg_rec_qa = sum(r for p, r in metrics_qa) / len(risultati_test)

print(f"--- RISULTATI FINALI PER IL REPORT ---")
print(f"PIPELINE TEXT-TO-SQL: Precision {avg_prec_sql:.2f}, Recall {avg_rec_sql:.2f}")
print(f"PIPELINE TABLE-QA:    Precision {avg_prec_qa:.2f}, Recall {avg_rec_qa:.2f}")
